In [ ]:
%pip install matplotlib numpy opencv-python

In [9]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

def listdir(path):
    """Lista archivos en un directorio"""
    return [os.path.join(path, file) for file in os.listdir(path)]

# Cargar imágenes
images = [(cv2.imread(image_path), image_path) for image_path in listdir('./img')]

# Función para mostrar imágenes en Matplotlib
def show_image(img, title="Imagen"):
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

# --- Filtros individuales ---
def apply_gaussian_blur(img):
    """Aplica desenfoque Gaussiano"""
    blurred = cv2.GaussianBlur(img, (5, 5), cv2.BORDER_DEFAULT)
    show_image(blurred, "Desenfoque Gaussiano")
    return blurred

def apply_otsu_threshold(img):
    """Aplica umbralización de Otsu"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, th3 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    show_image(th3, "Threshold de Otsu")
    return th3

def apply_median_filter(img):
    """Aplica filtro de Mediana"""
    median = cv2.medianBlur(img, 5)
    show_image(median, "Filtro de Mediana")
    return median

def apply_mean_smoothing(img):
    """Aplica un filtro de suavizado con un kernel Gaussiano"""
    kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
    smoothed = cv2.filter2D(img, -1, kernel)
    show_image(smoothed, "Filtro de Suavizado")
    return smoothed

def apply_CLAHE(img):
    """Aplica ecualización de histograma adaptativa (CLAHE)"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    equalized = clahe.apply(gray)
    show_image(equalized, "Ecualización CLAHE")
    return equalized

def adjust_contrast_brightness(img, alpha=1.5, beta=30):
    """
    Ajusta el contraste y brillo de la imagen.
    alpha > 1 aumenta el contraste, beta > 0 aumenta el brillo.
    """
    adjusted = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    show_image(adjusted, "Ajuste de Contraste y Brillo")
    return adjusted

# --- Función para aplicar filtros según el nombre de la imagen ---
def preprocess(img, path):
    """
    Aplica filtros específicos según el nombre del archivo.
    """
    filename = os.path.basename(path).lower()
    
    if "gauss" in filename:
        img = apply_gaussian_blur(img)
    if "otsu" in filename:
        img = apply_otsu_threshold(img)
    if "median" in filename:
        img = apply_median_filter(img)
    if "smooth" in filename:
        img = apply_mean_smoothing(img)
    if "clahe" in filename:
        img = apply_CLAHE(img)
    if "contrast" in filename:
        img = adjust_contrast_brightness(img)
    
    # Guardar imagen procesada
    os.makedirs("./out", exist_ok=True)
    cv2.imwrite(f"./out/{filename}", img)

# --- Procesar imágenes ---
for img, path in images:
    preprocess(img, path)
